# PawHealthAI - Vitality Score Engine v0.2

A runnable walkthrough of the PHAIVIT engine: the 0-100 wellbeing score, the red-flag
safety net, and the weight subsystem, ending with the full fixture harness.

**This notebook imports `phavit/` rather than redefining the engine.** An earlier version
carried its own copy of the code, which drifted out of step with the package and went
stale. Importing means what you see here is exactly what the tests and the product run.

The engine is **non-diagnostic**: it never names a disease and never tells an owner their
pet is healthy or sick. Every threshold below is a provisional engineering placeholder
pending veterinary review.

In [ ]:
import json
import sys
from datetime import date, datetime, timedelta
from pathlib import Path

# The package lives at the repo root. Works whether the kernel starts here or in test_case/.
ROOT = Path.cwd()
if not (ROOT / "phavit").is_dir() and (ROOT.parent / "phavit").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from phavit import (
    PetProfile, CheckInData, WeightReading, WeightTrendResult,
    process_checkin, compute_weight_trend, trend_from_checkins,
    declining_strength, signal_quality, body_status,
    BASE_PROFILES,
)
from phavit.weight import weight_penalty, staleness_factor
from phavit.scoring import (
    DEVIATION_WEIGHT, BODY_STATUS_ACTIVITY_BUMP, BODY_STATUS_APPETITE_BUMP,
    ACTIVITY_DROP_THRESHOLD_HEAVY,
)

TEST_DIR = ROOT / "test_case"
print("engine loaded from:", ROOT / "phavit")
print("fixtures:", len(list(TEST_DIR.glob("*.json"))))

## Helpers

Small builders so each example below states only what it is actually about.

In [ ]:
CHECKIN_DAY = datetime(2026, 7, 20)      # fixed, never `today` - weight staleness is date-dependent
LAST_WEIGH_IN = date(2026, 7, 18)


def make_pet(name="Scout", breed="Labrador", species="dog", sex="male",
             high_energy=False, worming=True, plan=False, birth=date(2020, 1, 1)):
    return PetProfile(
        pet_id="pet_demo", name=name, species=species, breed=breed, sex=sex,
        post_code=3000, birth_date=birth, is_high_energy=high_energy,
        worming_compliant=worming, vaccination_current=True,
        weight_management_plan=plan,
    )


def mk_checkin(ts=CHECKIN_DAY, a=4, e=4, s=4, m=60, **kw):
    return CheckInData(
        pet_id="pet_demo", timestamp=ts, appetite_score=a, energy_score=e,
        sleep_quality_score=s, activity_minutes=m, **kw
    )


def make_history(n=4, a=4, e=4, s=4, m=60):
    """n stable check-ins, one per week, ending the week before CHECKIN_DAY."""
    return [mk_checkin(CHECKIN_DAY - timedelta(weeks=n - i), a=a, e=e, s=s, m=m)
            for i in range(n)]


def show(label, result):
    score = result.score if result.score is not None else "withheld (override active)"
    print("-" * 68)
    print(f"  {label}")
    print("-" * 68)
    print(f"  Score       : {score}")
    print(f"  Band        : {result.band}")
    print(f"  Confidence  : {result.confidence}")
    print(f"  Drivers     : {', '.join(result.drivers)}")
    if result.override:
        rf = result.override
        print(f"  Red flag    : [{rf.severity_tier.upper()}] {rf.trigger}")
    print(f"  Explanation : {result.explanation}")
    print()

## Layer 1 - the wellbeing score

The score compares this week against **the pet's own recent baseline**, not against other
pets. Only downward moves cost points; a pet who is unusually lively simply stays at 100.

The first score unlocks on the **fourth** completed check-in - the current one counts, so
3 previous + current = 4.

In [ ]:
pet = make_pet()

# Too little history -> an honest "building baseline" state, not a fabricated number.
show("2 previous + current = 3 check-ins", process_checkin(pet, make_history(2), mk_checkin()))

# Fourth check-in unlocks the score.
show("3 previous + current = 4 check-ins", process_checkin(pet, make_history(3), mk_checkin()))

# A stable week against a stable baseline.
show("Stable week", process_checkin(pet, make_history(5), mk_checkin()))

# Appetite and energy each down one point: two indicators moving together.
show("Appetite 4->3 and energy 4->3",
     process_checkin(pet, make_history(5), mk_checkin(a=3, e=3)))

## Layer 2 - the red-flag safety net

Red flags run **before** the score and sit outside the 0-100 maths, so a serious sign can
never be averaged away by otherwise good vitals. Emergency and urgent flags withhold the
reassuring number entirely; a monitor flag is attached as advisory and the score still shows.

In [ ]:
show("Emergency - collapse",
     process_checkin(pet, make_history(5), mk_checkin(collapse=True)))

show("Urgent - blood in stool with low energy",
     process_checkin(pet, make_history(5), mk_checkin(e=2, blood_in_stool=True)))

show("Monitor - single vomiting (score still shown)",
     process_checkin(pet, make_history(5), mk_checkin(vomiting=True)))

## The weight subsystem

Weight runs on a **different clock** from the rest of the check-in. Owners weigh a pet
roughly monthly while wellbeing is reported weekly, so weight cannot be treated as just
another 1-5 score. Three rules keep a monthly signal from distorting a weekly number:

- **Carry-forward** - the stored *trend*, not the raw reading, applies to every check-in
  until the next weigh-in, so the score is steady between readings.
- **Staleness decay** - full strength for 8 weeks, fading linearly to nothing at 12.
- **Rate normalisation** - everything is expressed as **% of body weight lost per 4 weeks**,
  so readings taken at irregular intervals are comparable.

`weight.py` imports only from `models`. Scoring and red flags both *consume* its result, so
there is one definition of "what counts as a weight loss" rather than two that can drift.

In [ ]:
def monthly(kgs, last_on=LAST_WEIGH_IN):
    """Readings four weeks apart, ending on `last_on`."""
    n = len(kgs)
    return [WeightReading(kg=kg, at=last_on - timedelta(weeks=4 * (n - 1 - i)))
            for i, kg in enumerate(kgs)]


# A Labrador drifting down over four monthly weigh-ins.
trend = compute_weight_trend(
    monthly([32.0, 31.5, 31.0, 29.5]),
    species="dog", breed="Labrador", sex="male", birth_date=date(2020, 1, 1),
)

for f in ("status", "direction", "pct_change", "rate_per_4w", "baseline_kg",
          "current_kg", "as_of", "weeks_span", "body_status", "is_growing",
          "managed", "tier"):
    print(f"  {f:14s} {getattr(trend, f)}")
print("\n  notes:", trend.notes or "(none)")

### Screening comes first

Nothing reaches the maths unscreened. Values outside species plausibility bounds are
dropped, and a jump of more than 25% between consecutive readings is rejected as a
data-entry error - a mistyped 4.0 for 40.0 must never reach the escalation rules. When it
is the *most recent* reading that fails, the engine says so and asks the owner to re-check
rather than quietly scoring an older weight.

The comparison baseline is the **median** of up to four previous readings, not the mean, so
one bad entry cannot drag it.

In [ ]:
bad = compute_weight_trend(
    monthly([32.0, 31.5, 31.0, 3.10]),      # last reading: 31.0 mistyped as 3.10
    species="dog", breed="Labrador", sex="male", birth_date=date(2020, 1, 1),
)
print("  status :", bad.status, " -> the owner is asked to re-check, not scored")
print("  quality:", signal_quality(bad, CHECKIN_DAY))
for n in bad.notes:
    print("   -", n)

### How much a loss costs

Points are removed on a ramp: nothing below 2%/4wk, rising to a maximum of 15 points at
10%/4wk, where the safety net takes over instead so the deduction stops growing.

Note the **two different thresholds**. A loss deducts points from 2% upward, but only
counts towards the "several indicators declining together" tally from 5%. Small losses
should cost a little; only a *notable* loss should mark weight as one of several
indicators declining at once.

In [ ]:
def fake_trend(rate, as_of=LAST_WEIGH_IN, **kw):
    d = dict(status="ok", direction="loss", pct_change=rate, rate_per_4w=rate,
             baseline_kg=32.0, current_kg=30.0, as_of=as_of, body_status="in_reference")
    d.update(kw)
    return WeightTrendResult(**d)


print(f"  {'loss %/4wk':>11} | {'points off':>10} | counts towards 'several declining'")
print("  " + "-" * 58)
for r in (2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0):
    pts = weight_penalty(fake_trend(r), CHECKIN_DAY)
    tally = declining_strength(fake_trend(r), CHECKIN_DAY)
    # 3dp: the ramp lands on exact quarters (5% -> 5.625), and 2dp would hide that.
    shown = "0" if pts == 0 else f"{-pts:.3f}"
    print(f"  {r:>10.0f}% | {shown:>10} | {'yes' if tally else 'no'}")

In [ ]:
# Staleness: the same 10%/4wk loss, seen at increasing distance from the weigh-in.
print(f"  {'weeks since weigh-in':>21} | {'points off':>10} | {'compound share':>14} | quality")
print("  " + "-" * 70)
for wk in (0, 4, 8, 9, 10, 11, 12, 16):
    at = datetime.combine(LAST_WEIGH_IN, datetime.min.time()) + timedelta(weeks=wk)
    t = fake_trend(10.0)
    print(f"  {wk:>21} | {-weight_penalty(t, at):>10.2f} | "
          f"{declining_strength(t, at):>14.2f} | {signal_quality(t, at)}")

The decay is deliberately gradual. Weight contributes to the compound tally as a
**fraction** rather than a yes/no: with a hard cut-off, a signal one day short of 12 weeks
carried its full compound weight and the next week carried none, moving the score by
double figures on no new data.

Because the score quietly **rises** as an old signal decays, the engine says so out loud -
otherwise an owner sees an improving number with no way to know it reflects an expiring
reading rather than a recovering pet.

## Weight inside the score

Same pet, same check-in, four different weight situations.

In [ ]:
history = make_history(5)

show("No weight ever recorded (engine behaves exactly as before weight existed)",
     process_checkin(pet, history, mk_checkin()))

show("Gradual loss, ~6%/4wk, fresh reading",
     process_checkin(pet, history, mk_checkin(), weight_trend=fake_trend(6.0)))

show("Same loss, but the reading is 10 weeks old",
     process_checkin(pet, history, mk_checkin(),
                     weight_trend=fake_trend(6.0, as_of=CHECKIN_DAY.date() - timedelta(weeks=10))))

show("Rejected weight entry - a prompt, not a deduction",
     process_checkin(pet, history, mk_checkin(),
                     weight_trend=WeightTrendResult(status="implausible", current_kg=3.1,
                                                    as_of=LAST_WEIGH_IN)))

## Weight inside the safety net

Three weight rules can escalate. Two are urgent, one is advisory only.

| Trigger | Tier | Fires when |
|---|---|---|
| `notable_weight_loss` | urgent | unmanaged loss >= 10%/4wk (>= 7% while growing) |
| `weight_loss_with_signs` | urgent | loss >= 5%/4wk **plus** low appetite, increased thirst, or low energy |
| `rapid_weight_gain` | monitor | gain >= 8%/4wk in an adult; suppressed entirely while growing |

`weight_loss_with_signs` exists because each of those companion signs needs *repetition* to
escalate on its own. A first occurrence next to real weight loss would otherwise slip through.

In [ ]:
show("Urgent - notable loss on its own",
     process_checkin(pet, history, mk_checkin(),
                     weight_trend=fake_trend(12.0, tier="urgent", pct_change=12.0)))

show("Urgent - a 6% loss plus reduced appetite on this check-in",
     process_checkin(pet, history, mk_checkin(a=2), weight_trend=fake_trend(6.0)))

show("Monitor - rapid gain is advisory, the score still shows",
     process_checkin(pet, history, mk_checkin(),
                     weight_trend=fake_trend(-9.0, direction="gain", tier="monitor",
                                             pct_change=-9.0)))

### Two deliberate suppressions

A pet on a recorded **weight-management plan** is not penalised for losing weight at a safe
rate - owners doing the right thing should not be alarmed for succeeding. Loss faster than
the ceiling (8%/4wk for dogs, 4% for cats) escalates anyway.

A **growing** pet has gain rules switched off entirely and loss thresholds tightened by 30%,
because a growing animal losing weight matters more.

In [ ]:
planned = make_pet(plan=True)
t_planned = compute_weight_trend(
    monthly([34.0, 33.0, 32.0, 31.2]),
    species="dog", breed="Labrador", sex="male", birth_date=date(2020, 1, 1),
    weight_management_plan=True,
)
print(f"  managed plan  : rate {t_planned.rate_per_4w}%/4wk, managed={t_planned.managed}, "
      f"tier={t_planned.tier}, points off={weight_penalty(t_planned, CHECKIN_DAY):.2f}")

puppy = compute_weight_trend(
    monthly([10.0, 11.0, 12.0]),
    species="dog", breed="Labrador", sex="male", birth_date=date(2025, 10, 1),
)
print(f"  growing puppy : direction={puppy.direction}, is_growing={puppy.is_growing}, "
      f"tier={puppy.tier}  (gain rules suppressed)")
print(f"  body status   : {puppy.body_status}  (a growing animal has no adult range to compare to)")

## Weighting profiles

The base profile comes from the breed's energy level. Where the pet sits against its breed
weight reference then shifts the emphasis, after which the weights are **renormalised back
to a sum of 1.0**.

That renormalisation is load-bearing: bumping activity without rescaling pushes the total
above 1.0, which makes the score uniformly *harsher* rather than simply re-prioritising
activity - a different, and worse, behaviour.

For a pet carrying extra weight, sustained low activity is both more consequential and the
lever an owner can actually pull, so activity is weighted up and the bar for calling it a
"drop" falls from 25% to 20%. For a pet *below* its reference the lever is intake rather
than exercise, so appetite is weighted up instead. Neither is a deduction in itself.

In [ ]:
from dataclasses import replace

print(f"  {'profile':38s} {'appet':>6} {'energy':>7} {'sleep':>6} {'activ':>6} {'sum':>6} {'drop@':>6}")
print("  " + "-" * 84)
for pname in ("standard", "high_energy"):
    base = BASE_PROFILES[pname]
    for bs in (None, "above_reference", "well_above_reference", "below_reference"):
        ab = BODY_STATUS_ACTIVITY_BUMP.get(bs, 0.0)
        pb = BODY_STATUS_APPETITE_BUMP.get(bs, 0.0)
        w = base if not (ab or pb) else replace(
            base, activity=base.activity + ab, appetite=base.appetite + pb,
            activity_drop_threshold=(ACTIVITY_DROP_THRESHOLD_HEAVY if ab
                                     else base.activity_drop_threshold),
        ).normalised()
        total = w.appetite + w.energy + w.sleep + w.activity
        assert abs(total - 1.0) < 1e-9, f"{pname}/{bs} does not sum to 1.0"
        label = f"{pname}, {bs or 'no reference shift'}"
        print(f"  {label:38s} {w.appetite:6.3f} {w.energy:7.3f} {w.sleep:6.3f} "
              f"{w.activity:6.3f} {total:6.3f} {w.activity_drop_threshold:6.0%}")
print("\n  every profile sums to 1.0")

In [ ]:
# What those weights mean in points - a one-point drop, by profile.
print(f"  {'profile':38s} {'appetite -1':>12} {'energy -1':>10} {'sleep -1':>9} {'activity -50%':>14}")
print("  " + "-" * 86)
for pname in ("standard", "high_energy"):
    base = BASE_PROFILES[pname]
    for bs in (None, "above_reference", "well_above_reference", "below_reference"):
        ab = BODY_STATUS_ACTIVITY_BUMP.get(bs, 0.0)
        pb = BODY_STATUS_APPETITE_BUMP.get(bs, 0.0)
        w = base if not (ab or pb) else replace(
            base, activity=base.activity + ab, appetite=base.appetite + pb).normalised()
        label = f"{pname}, {bs or 'no reference shift'}"
        print(f"  {label:38s} {-w.appetite * DEVIATION_WEIGHT:12.2f} "
              f"{-w.energy * DEVIATION_WEIGHT:10.2f} {-w.sleep * DEVIATION_WEIGHT:9.2f} "
              f"{-0.5 * w.activity * DEVIATION_WEIGHT:14.2f}")

## Weight arriving on the check-in stream

Weights reach the engine two ways. If the product stores a weight log, the trend is
computed once at weigh-in and passed to `process_checkin()`. If it does not, the engine
derives the trend from the optional `weight_kg` field on the check-ins themselves - so an
owner who weighs their pet occasionally gets the benefit without a separate log.

A pet with no recorded weight at all yields `None`, and the engine behaves exactly as it
did before weight existed.

In [ ]:
# Weekly check-ins, most carrying no weight at all - which is the normal case.
weekly = []
weights = {0: 32.0, 4: 31.0, 8: 29.8}
for wk in range(9):
    kw = {"weight_kg": weights[wk]} if wk in weights else {}
    weekly.append(mk_checkin(CHECKIN_DAY - timedelta(weeks=8 - wk), **kw))

derived = trend_from_checkins(pet, weekly)
print(f"  derived from check-ins : {derived.direction} {derived.rate_per_4w}%/4wk "
      f"as of {derived.as_of}")

no_weight = trend_from_checkins(pet, make_history(5))
print(f"  no weight recorded     : {no_weight}   <- engine falls back to pre-weight behaviour")

# The pipeline does this for you when no trend is supplied.
show("Derived automatically inside process_checkin()",
     process_checkin(pet, weekly[:-1], weekly[-1]))

## The fixture harness

Every scenario in `test_case/*.json` declares its expected band, override tier and weight
outcome **before** the engine runs. Files are globbed automatically, so a new scenario
needs no registration anywhere.

These are the same loaders the test suite uses, typo tolerance included - several fixtures
carry `appetite_scorew` and `diarrhea`, and the loaders fall back rather than crash.

In [ ]:
def load_pet_profile(p):
    return PetProfile(
        pet_id=p["pet_id"],
        name=p["pet_name"],                       # JSON uses pet_name
        species=p["species"], breed=p["breed"], sex=p["sex"],
        post_code=int(p["postcode"]),             # JSON uses postcode
        birth_date=date.fromisoformat(p["birth_date"]),
        is_high_energy=p["is_high_energy"],
        worming_compliant=p["worming_compliant"],
        vaccination_current=p["vaccination_current"],
        weight_management_plan=p.get("weight_management_plan", False),
    )


def load_checkin(c):
    return CheckInData(
        pet_id=c["pet_id"],
        timestamp=datetime.fromisoformat(c["timestamp"].replace("Z", "+00:00")),
        appetite_score=c.get("appetite_score", c.get("appetite_scorew")),   # typo fallback
        energy_score=c["energy_score"],
        sleep_quality_score=c["sleep_quality_score"],
        activity_minutes=c["activity_minutes"],
        toileting_status=c.get("toileting_status", "normal"),
        water_intake_status=c.get("water_intake_status", "normal"),
        blood_in_stool=c.get("blood_in_stool", False),
        vomiting=c.get("vomiting", False),
        diarrhoea=c.get("diarrhoea", c.get("diarrhea", False)),             # typo fallback
        breathing_difficulty=c.get("breathing_difficulty", False),
        collapse=c.get("collapse", False),
        seizure=c.get("seizure", False),
        suspected_toxin_ingestion=c.get("suspected_toxin_ingestion", False),
        unable_to_urinate=c.get("unable_to_urinate", False),
        severe_bleeding=c.get("severe_bleeding", False),
        tick_found=c.get("tick_found", False),
        pain_or_discomfort_signs=c.get("pain_or_discomfort_signs", False),
        weight_kg=c.get("weight_kg"),
        owner_concern_level=c.get("owner_concern_level"),
    )


def load_previous(data, pet_id):
    """Omitted history means a synthetic stable baseline - NOT an empty list."""
    raw = data.get("previous_checkins")
    if raw is None:
        return [CheckInData(pet_id=pet_id, timestamp=datetime(2026, 1, i + 1),
                            appetite_score=4, energy_score=4,
                            sleep_quality_score=4, activity_minutes=60) for i in range(4)]
    return [load_checkin(c) for c in raw]


def load_weight_trend(data, pet):
    raw = (data.get("weight_log") or {}).get("readings")
    if not raw:
        return None      # fall through to the check-in stream
    readings = [WeightReading(kg=float(r["kg"]), at=date.fromisoformat(r["at"])) for r in raw]
    return compute_weight_trend(
        readings, species=pet.species, breed=pet.breed, sex=pet.sex,
        birth_date=pet.birth_date, weight_management_plan=pet.weight_management_plan,
    )


def run_scenario(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    pet = load_pet_profile(data["pet_profile"])
    result = process_checkin(
        pet, load_previous(data, pet.pet_id), load_checkin(data["check_in_data"]),
        weight_trend=load_weight_trend(data, pet),
    )
    return data, pet, result


print("loaders ready")

In [ ]:
# Run every fixture and check it against its own declared expectation.
FIELD_CHECKS = {
    "band": lambda r: r.band,
    "score": lambda r: r.score,
    "confidence": lambda r: r.confidence,
    "trend": lambda r: r.trend,
    "baseline_status": lambda r: r.baseline_status,
    "override_tier": lambda r: r.override.severity_tier if r.override else None,
    "override_trigger": lambda r: r.override.trigger if r.override else None,
    "weight_status": lambda r: r.weight_trend.status if r.weight_trend else None,
    "weight_direction": lambda r: r.weight_trend.direction if r.weight_trend else None,
    "weight_body_status": lambda r: r.weight_trend.body_status if r.weight_trend else None,
    "weight_managed": lambda r: r.weight_trend.managed if r.weight_trend else None,
    "weight_is_growing": lambda r: r.weight_trend.is_growing if r.weight_trend else None,
}

passed, failures = 0, []
for path in sorted(TEST_DIR.glob("*.json")):
    data, pet, result = run_scenario(path)
    exp = data.get("expected", {})
    problems = []
    for key, get in FIELD_CHECKS.items():
        if key in exp and get(result) != exp[key]:
            problems.append(f"{key}: got {get(result)!r}, expected {exp[key]!r}")
    if exp.get("score_present") is not None:
        if (result.score is not None) != exp["score_present"]:
            problems.append(f"score_present: got {result.score is not None}")
    for frag in exp.get("drivers_include", []):
        if not any(frag in d for d in result.drivers):
            problems.append(f"missing driver fragment {frag!r}")
    if problems:
        failures.append((path.name, problems))
    else:
        passed += 1

print(f"{passed}/{passed + len(failures)} fixtures match their declared expectation\n")
for name, problems in failures:
    print(f"  FAIL {name}")
    for p in problems:
        print(f"       {p}")

### The non-diagnostic guard

The engine's most important safety property: **no disease name ever reaches an owner.**
This runs across every fixture, on every user-facing string.

Note it deliberately does not ban the words "diagnosis" or "disease" - the engine uses them
in its safe disclaimers ("this is a wellbeing trend, not a diagnosis").

In [ ]:
DISEASE_WORDS = [
    "cancer", "tumour", "tumor", "diabetes", "kidney disease", "renal failure",
    "pancreatitis", "parvo", "arthritis", "hyperthyroid", "hypothyroid",
    "cushing", "addison", "lymphoma", "infection", "poisoning", "obstruction",
]

leaks = []
for path in sorted(TEST_DIR.glob("*.json")):
    data, pet, result = run_scenario(path)
    surfaces = [result.explanation, result.band, *result.drivers]
    if result.override:
        surfaces += [result.override.clinical_reason_for_escalation,
                     result.override.recommended_user_pathway]
    for text in filter(None, surfaces):
        for word in DISEASE_WORDS:
            if word in text.lower():
                leaks.append((path.name, word, text))

print(f"checked {len(list(TEST_DIR.glob('*.json')))} fixtures")
print("disease terms leaked:", len(leaks))
for name, word, text in leaks:
    print(f"  {name}: {word!r} in {text!r}")

## Where to go next

- `test_case/README.md` - how to run the 92-test suite and how to add to it
- `vitality_score_engine_v0_2_report.md` - the technical report, including the formal
  notation and the open questions for veterinary review
- `PawHealthAI_Vitality_Score.md` - the product-level overview

Every threshold here is a provisional engineering placeholder. Where the literature and a
constant disagree, the literature wins and the constant should change.